# PART 03 · 3장 분류 모델링

교재 238~259쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

### 1 공통 준비 코드

#### 1. 공통 라이브러리 불러오기

이 장에서 공통으로 사용하는 라이브러리를 한 번에 불러온다.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix

#### 2. 이진분류 데이터 준비 1

이진분류 데이터를 불러와 target을 분리하고 학습용과 검증용으로 나눈다.

In [2]:
binary_base = 'data/part03_type2/binary_learning_renewal'
binary_train = pd.read_csv(binary_base+'/train.csv')
binary_test = pd.read_csv(binary_base+'/test.csv')
binary_submit = pd.read_csv(binary_base+'/sample_submission.csv')
y_bin = binary_train['renew_next_month'].copy()
X_bin_raw = binary_train.drop(
    columns=['renew_next_month', 'student_id']
).copy()
X_bin_test_raw = binary_test.drop(columns=['student_id']).copy()
X_bin_train_raw, X_bin_val_raw, y_bin_train, y_bin_val = train_test_split(
    X_bin_raw, y_bin, test_size=0.2, random_state=0, stratify=y_bin
)

#### 3. 이진분류 데이터 준비 2

결측치 처리와 인코딩까지 마쳐 이진분류 입력을 완성한다.

In [3]:
num_cols = X_bin_train_raw.select_dtypes(exclude='object').columns
cat_cols = X_bin_train_raw.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_bin_train_raw[col].median()
    X_bin_train_raw[col] = X_bin_train_raw[col].fillna(fill_value)
    X_bin_val_raw[col] = X_bin_val_raw[col].fillna(fill_value)
    X_bin_test_raw[col] = X_bin_test_raw[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_bin_train_raw[col].mode()[0]
    X_bin_train_raw[col] = X_bin_train_raw[col].fillna(fill_value)
    X_bin_val_raw[col] = X_bin_val_raw[col].fillna(fill_value)
    X_bin_test_raw[col] = X_bin_test_raw[col].fillna(fill_value)
X_bin_train = pd.get_dummies(X_bin_train_raw)
X_bin_val = pd.get_dummies(X_bin_val_raw).reindex(
    columns=X_bin_train.columns, fill_value=0
)
X_bin_test = pd.get_dummies(X_bin_test_raw).reindex(
    columns=X_bin_train.columns, fill_value=0
)

#### 4. 다중분류 데이터 준비 1

다중분류 데이터도 같은 방식으로 불러와 분할한다.

In [4]:
multi_base = 'data/part03_type2/multiclass_ticket_priority'
multi_train = pd.read_csv(multi_base+'/train.csv')
multi_test = pd.read_csv(multi_base+'/test.csv')
multi_submit = pd.read_csv(multi_base+'/sample_submission.csv')
y_multi = multi_train['priority_level'].copy()
X_multi_raw = multi_train.drop(
    columns=['priority_level', 'ticket_id']
).copy()
X_multi_test_raw = multi_test.drop(columns=['ticket_id']).copy()
X_multi_train_raw, X_multi_val_raw, y_multi_train, y_multi_val = train_test_split(
    X_multi_raw, y_multi, test_size=0.2, random_state=0, stratify=y_multi
)

#### 5. 다중분류 데이터 준비 2

결측치 처리와 인코딩까지 마쳐 다중분류 입력을 완성한다.

In [5]:
num_cols = X_multi_train_raw.select_dtypes(exclude='object').columns
cat_cols = X_multi_train_raw.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_multi_train_raw[col].median()
    X_multi_train_raw[col] = X_multi_train_raw[col].fillna(fill_value)
    X_multi_val_raw[col] = X_multi_val_raw[col].fillna(fill_value)
    X_multi_test_raw[col] = X_multi_test_raw[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_multi_train_raw[col].mode()[0]
    X_multi_train_raw[col] = X_multi_train_raw[col].fillna(fill_value)
    X_multi_val_raw[col] = X_multi_val_raw[col].fillna(fill_value)
    X_multi_test_raw[col] = X_multi_test_raw[col].fillna(fill_value)
X_multi_train = pd.get_dummies(X_multi_train_raw)
X_multi_val = pd.get_dummies(X_multi_val_raw).reindex(
    columns=X_multi_train.columns, fill_value=0
)
X_multi_test = pd.get_dummies(X_multi_test_raw).reindex(
    columns=X_multi_train.columns, fill_value=0
)

#### 6. 준비된 데이터 구조 확인

준비된 두 문제의 데이터 구조와 target 분포를 확인해본다.

In [6]:
print('이진분류: ', X_bin_train.shape, X_bin_val.shape, X_bin_test.shape)
print(y_bin.value_counts().sort_index())
print()
print('다중분류: ', X_multi_train.shape, X_multi_val.shape, X_multi_test.shape)
print(y_multi.value_counts().sort_index())

이진분류:  (576, 28) (144, 28) (240, 28)
renew_next_month
0    332
1    388
Name: count, dtype: int64

다중분류:  (624, 28) (156, 28) (260, 28)
priority_level
0    304
1    359
2    117
Name: count, dtype: int64


## 01 분류 모델링

## 02 이진분류 문제 처리

### 1 이진분류 확인 사항

#### 1. 이진분류에서 먼저 확인할 내용

#### 1. target과 제출 구조 확인

target 분포와 제출 파일의 구조를 확인해본다.

In [7]:
print(y_bin.value_counts().sort_index())
print(binary_submit.shape)
print(binary_submit.head())

renew_next_month
0    332
1    388
Name: count, dtype: int64
(240, 1)
   pred
0   0.0
1   0.0
2   0.0
3   0.0
4   0.0


### 2 기본 풀이 흐름

학습, 예측, 평가로 이어지는 기본 흐름을 한 번에 실행해본다.

In [8]:
binary_model = RandomForestClassifier(n_estimators=100, random_state=0)
binary_model.fit(X_bin_train, y_bin_train)
binary_pred = binary_model.predict(X_bin_val)
positive_index = list(binary_model.classes_).index(1)
binary_prob = binary_model.predict_proba(X_bin_val)[:, positive_index]
print('ROC-AUC: ', round(roc_auc_score(y_bin_val, binary_prob), 4))
print('Accuracy: ', round(accuracy_score(y_bin_val, binary_pred), 4))
print('클래스 값: ', binary_pred[:5])
print('1클래스 확률: ', binary_prob[:5])

ROC-AUC:  0.7061
Accuracy:  0.6181
클래스 값:  [1 1 1 0 1]
1클래스 확률:  [0.81 0.51 0.76 0.28 0.52]


## 03 다중분류 문제 처리

#### 1. target과 제출 구조 확인

다중분류 데이터의 크기와 target 분포, 제출 구조를 확인해본다.

In [9]:
print(multi_train.shape, multi_test.shape, multi_submit.shape)
print(y_multi.value_counts().sort_index())
print(multi_submit.head())

(780, 15) (260, 14) (260, 1)
priority_level
0    304
1    359
2    117
Name: count, dtype: int64
   pred
0     0
1     0
2     0
3     0
4     0


#### 2. 기본 다중분류 흐름

다중분류도 기본 흐름은 같다. 모델을 학습하고 예측 클래스와 확률 배열을 확인해본다.

In [10]:
multi_model = RandomForestClassifier(random_state=0)
multi_model.fit(X_multi_train, y_multi_train)
multi_pred = multi_model.predict(X_multi_val)
multi_proba = multi_model.predict_proba(X_multi_val)
print(y_multi_val.value_counts().sort_index())
print('Accuracy: ', round(accuracy_score(y_multi_val, multi_pred), 4))
print('MacroF1: ', round(f1_score(y_multi_val, multi_pred, average='macro'), 4))
print('예측값: ', multi_pred[:10])
print('확률배열: ', multi_proba.shape)

priority_level
0    61
1    72
2    23
Name: count, dtype: int64
Accuracy:  0.5769
MacroF1:  0.4595
예측값:  [1 1 1 0 2 0 0 1 0 1]
확률배열:  (156, 3)


## 04 LogisticRegression

### 1 스케일링과 모델 학습

스케일링과 로지스틱 회귀 학습을 이어서 실행해본다.

In [11]:
scaler = StandardScaler()
X_bin_train_scaled = scaler.fit_transform(X_bin_train)
X_bin_val_scaled = scaler.transform(X_bin_val)
logistic_model = LogisticRegression(max_iter=1000, random_state=0)
logistic_model.fit(X_bin_train_scaled, y_bin_train)
positive_index = list(logistic_model.classes_).index(1)
logistic_prob = logistic_model.predict_proba(X_bin_val_scaled)[:, positive_index]
logistic_pred = logistic_model.predict(X_bin_val_scaled)
print(X_bin_train_scaled.shape, X_bin_val_scaled.shape)
print('ROC-AUC: ', round(roc_auc_score(y_bin_val, logistic_prob), 4))
print('Accuracy: ', round(accuracy_score(y_bin_val, logistic_pred), 4))
print(logistic_prob[:5].round(2))
print(logistic_pred[:5])

(576, 28) (144, 28)
ROC-AUC:  0.7164
Accuracy:  0.6389
[0.9  0.47 0.72 0.2  0.39]
[1 0 1 0 0]


### 2 주요 코드 해설

## 05 DecisionTreeClassifier

#### 1. 모델 학습과 검증

max_depth를 4로 제한한 결정나무를 학습하고 검증 점수를 확인해본다.

In [12]:
tree_model = DecisionTreeClassifier(max_depth=4, random_state=0)
tree_model.fit(X_bin_train, y_bin_train)
positive_index = list(tree_model.classes_).index(1)
tree_prob = tree_model.predict_proba(X_bin_val)[:, positive_index]
tree_pred = tree_model.predict(X_bin_val)
print('ROC-AUC: ', round(roc_auc_score(y_bin_val, tree_prob), 4))
print('Accuracy: ', round(accuracy_score(y_bin_val, tree_pred), 4))
print('확률: ', tree_prob[:5].round(2))
print('클래스: ', tree_pred[:5])
print('depth: ', tree_model.get_depth())
print('leaves: ', tree_model.get_n_leaves())

ROC-AUC:  0.6688
Accuracy:  0.5903
확률:  [0.72 0.52 0.72 0.12 0.23]
클래스:  [1 1 1 0 0]
depth:  4
leaves:  14


#### 2. max_depth의 역할

## 06 RandomForestClassifier

### 1 모델 학습과 검증

랜덤포레스트를 학습하고 같은 방식으로 검증 점수를 확인해본다.

In [13]:
forest_model = RandomForestClassifier(n_estimators=100, random_state=0)
forest_model.fit(X_bin_train, y_bin_train)
positive_index = list(forest_model.classes_).index(1)
forest_prob = forest_model.predict_proba(X_bin_val)[:, positive_index]
forest_pred = forest_model.predict(X_bin_val)
print('ROC-AUC: ', round(roc_auc_score(y_bin_val, forest_prob), 4))
print('Accuracy: ', round(accuracy_score(y_bin_val, forest_pred), 4))
print(forest_prob[:5])
print(forest_pred[:5])

ROC-AUC:  0.7061
Accuracy:  0.6181
[0.81 0.51 0.76 0.28 0.52]
[1 1 1 0 1]


### 2 모델 비교와 최종 예측

#### 1. 세 모델의 검증 결과 비교

#### 2. 모델 비교와 최종 test 확률 생성

비교 결과가 가장 좋은 모델을 선택해 전체 train으로 다시 학습하고 test 확률을 만들어 본다.

In [14]:
# 현재 비교표에서 ROC-AUC가 가장 높은 LogisticRegression을 선택한다.
X_bin_full = binary_train.drop(
    columns=['renew_next_month', 'student_id']
).copy()
X_bin_test_full = binary_test.drop(columns=['student_id']).copy()
y_bin_full = binary_train['renew_next_month'].copy()
num_cols = X_bin_full.select_dtypes(exclude='object').columns
cat_cols = X_bin_full.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_bin_full[col].median()
    X_bin_full[col] = X_bin_full[col].fillna(fill_value)
    X_bin_test_full[col] = X_bin_test_full[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_bin_full[col].mode()[0]
    X_bin_full[col] = X_bin_full[col].fillna(fill_value)
    X_bin_test_full[col] = X_bin_test_full[col].fillna(fill_value)
X_bin_full = pd.get_dummies(X_bin_full)
X_bin_test_full = pd.get_dummies(X_bin_test_full).reindex(
    columns=X_bin_full.columns, fill_value=0
)
final_scaler = StandardScaler()
X_bin_full_scaled = final_scaler.fit_transform(X_bin_full)
X_bin_test_scaled = final_scaler.transform(X_bin_test_full)
final_model = LogisticRegression(max_iter=1000, random_state=0)
final_model.fit(X_bin_full_scaled, y_bin_full)
positive_index = list(final_model.classes_).index(1)
test_prob = final_model.predict_proba(X_bin_test_scaled)[:, positive_index]
result = pd.DataFrame({'pred': test_prob})
print(result.shape)
print(result.head())

(240, 1)
       pred
0  0.321809
1  0.390323
2  0.551854
3  0.215861
4  0.156865


## 07 predict와 predict_proba

### 1 두 메서드의 차이

#### 1. 두 메서드의 차이

### 2 분류 유형별 결과 비교

#### 1. 이진분류 결과 비교

이진분류 모델에서 두 메서드의 결과 형태를 비교해본다.

In [15]:
val_pred = forest_model.predict(X_bin_val)
val_proba = forest_model.predict_proba(X_bin_val)
print('classes: ', forest_model.classes_)
print('predict_shape: ', val_pred.shape)
print('predict_proba.shape: ', val_proba.shape)
print('predict 앞 5개: ', val_pred[:5])
print(pd.DataFrame(
    val_proba[:5], columns=['0일 확률', '1일 확률']
).round(2))

classes:  [0 1]
predict_shape:  (144,)
predict_proba.shape:  (144, 2)
predict 앞 5개:  [1 1 1 0 1]
   0일 확률  1일 확률
0   0.19   0.81
1   0.49   0.51
2   0.24   0.76
3   0.72   0.28
4   0.48   0.52


#### 2. 다중분류의 확률 배열

다중분류에서는 확률 배열의 열이 클래스 수만큼 생긴다.

In [16]:
print('classes: ', multi_model.classes_)
print('predict_proba.shape: ', multi_proba.shape)
print(pd.DataFrame(
    multi_proba[:3], columns=multi_model.classes_
).round(2))

classes:  [0 1 2]
predict_proba.shape:  (156, 3)
      0     1     2
0  0.37  0.56  0.07
1  0.20  0.49  0.31
2  0.13  0.69  0.18


## 08 ROC-AUC

### 1 ROC-AUC

#### 1. ROC-AUC 계산

ROC-AUC는 관심 클래스의 확률로 계산한다. predict_proba( )는 클래스별 확률을 한꺼번에 반환하므로 classes_에서 라벨 1의 위치를 확인한 뒤 해당 열을 선택한다. 예측 클래스로 계산하는 Accuracy와 함께 확인해본다.

In [17]:
val_pred = forest_model.predict(X_bin_val)
positive_index = list(forest_model.classes_).index(1)
val_prob_1 = forest_model.predict_proba(X_bin_val)[:, positive_index]
print('1 클래스 확률: ', val_prob_1[:5])
print('Accuracy: ', round(accuracy_score(y_bin_val, val_pred), 4))
print('ROC-AUC: ', round(roc_auc_score(y_bin_val, val_prob_1), 4))

1 클래스 확률:  [0.81 0.51 0.76 0.28 0.52]
Accuracy:  0.6181
ROC-AUC:  0.7061


#### 2. 클래스값과 확률값으로 계산한 결과 비교

클래스값을 넣었을 때와 확률값을 넣었을 때의 ROC-AUC를 비교해본다.

In [18]:
auc_from_class = roc_auc_score(y_bin_val, val_pred)
auc_from_proba = roc_auc_score(y_bin_val, val_prob_1)
print(round(auc_from_class, 4))
print(round(auc_from_proba, 4))

0.6136
0.7061


### 2 기준값에 따른 예측 변화

#### 1. 기준값에 따른 예측 변화

## 09 Accuracy

### 1 Accuracy 계산

Accuracy를 직접 계산한 값과 accuracy_score( ) 결과를 비교해본다.

In [19]:
multi_pred = multi_model.predict(X_multi_val)
correct_count = (multi_pred == y_multi_val).sum()
total_count = len(y_multi_val)
manual_accuracy = correct_count/total_count
print('맞힌 개수: ', correct_count)
print('전체 개수: ', total_count)
print('직접계산: ', round(manual_accuracy, 4))
print('accuracy_score: ', round(
    accuracy_score(y_multi_val, multi_pred), 4
))

맞힌 개수:  90
전체 개수:  156
직접계산:  0.5769
accuracy_score:  0.5769


### 2 혼동행렬과 지표 비교

#### 1. 혼동행렬로 맞힌 위치 확인

#### 2. Accuracy와 다른 지표 비교

## 10 F1-score

### 1 F1-score의 구성

average=None을 지정해 클래스별 정밀도, 재현율, F1-score를 확인해본다.

In [20]:
f1_each = f1_score(y_multi_val, multi_pred, average=None)
precision_each = precision_score(
    y_multi_val, multi_pred, average=None, zero_division=0
)
recall_each = recall_score(
    y_multi_val, multi_pred, average=None, zero_division=0
)
print('정밀도: ', precision_each)
print('재현율: ', recall_each)
print('F1-score: ', f1_each)

정밀도:  [0.58730159 0.57303371 0.5       ]
재현율:  [0.60655738 0.70833333 0.08695652]
F1-score:  [0.59677419 0.63354037 0.14814815]


## 11 Macro F1

### 1 Macro F1 계산

클래스별 F1-score의 단순 평균이 Macro F1이다. 직접 계산한 값과 함수 결과를 비교해본다.

In [21]:
f1_each = f1_score(y_multi_val, multi_pred, average=None)
macro_f1 = f1_each.mean()
print(f1_each)
print(round(macro_f1, 4))
print(round(
    f1_score(y_multi_val, multi_pred, average='macro'), 4
))

[0.59677419 0.63354037 0.14814815]
0.4595
0.4595


### 2 지표 비교와 제출값 생성

다중분류 제출값을 만들어 본다. 전체 train으로 다시 학습해 test의 클래스값을 저장한다.

In [22]:
multi_X = multi_train.drop(columns = ['priority_level', 'ticket_id']).copy()
multi_test_X = multi_test.drop(columns=['ticket_id']).copy()
num_cols = multi_X.select_dtypes(include='number').columns
cat_cols = multi_X.select_dtypes(exclude='number').columns
for col in num_cols:
    fill_value = multi_X[col].median()
    multi_X[col] = multi_X[col].fillna(fill_value)
    multi_test_X[col] = multi_test_X[col].fillna(fill_value)
for col in cat_cols:
    fill_value = multi_X[col].mode()[0]
    multi_X[col] = multi_X[col].fillna(fill_value)
    multi_test_X[col] = multi_test_X[col].fillna(fill_value)
multi_X = pd.get_dummies(multi_X)
multi_test_X = pd.get_dummies(multi_test_X).reindex(
    columns = multi_X.columns,
    fill_value = 0
)
final_multi_model = RandomForestClassifier(random_state=0)
final_multi_model.fit(multi_X, y_multi)
test_multi_pred = final_multi_model.predict(multi_test_X)
result = pd.DataFrame({'pred': test_multi_pred})
print(result.shape)
print(result.head())
print(result['pred'].value_counts().sort_index())

(260, 1)
   pred
0     0
1     0
2     1
3     1
4     1
pred
0    103
1    152
2      5
Name: count, dtype: int64
